In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint04b_Regime_And_Shorts

Purpose:
Add a market regime filter and short-side positions to all four
strategies, and compare long-only vs regime-gated long/short.

Author:
Alison

Version:
0.6
"""

In [ ]:
# =====================================================
# ALPHA v0.6
# Sprint 4b - Regime Filter + Long/Short
# =====================================================

import matplotlib.pyplot as plt
import pandas as pd

from alpha.config import DEFAULT_CONFIG
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.portfolio import calculate_monthly_returns, build_portfolio, build_long_short_portfolio
from alpha.regime import calculate_regime, apply_regime_filter

from alpha.strategies.momentum import calculate_monthly_momentum, select_top_stocks, select_bottom_stocks
from alpha.strategies.trend_following import select_trend_positions, select_downtrend_positions
from alpha.strategies.mean_reversion import select_oversold_stocks, select_overbought_stocks
from alpha.strategies.breakout import select_breakout_stocks, select_breakdown_stocks

In [ ]:
config = DEFAULT_CONFIG
config

In [ ]:
# =====================================================
# DATA
# =====================================================

print("Downloading market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)
monthly_returns = calculate_monthly_returns(monthly_prices)

print("Downloading benchmark for regime filter...")

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

print(f"Bull months: {regime.sum()} / {len(regime)}")

In [ ]:
# =====================================================
# LONG AND SHORT SIGNALS FOR EACH STRATEGY
# =====================================================

monthly_momentum = calculate_monthly_momentum(monthly_prices, config)

signal_pairs = {
    "Momentum": (
        select_top_stocks(monthly_momentum, config),
        select_bottom_stocks(monthly_momentum, config),
    ),
    "Trend Following": (
        select_trend_positions(monthly_prices, config),
        select_downtrend_positions(monthly_prices, config),
    ),
    "Mean Reversion": (
        select_oversold_stocks(monthly_prices, config),
        select_overbought_stocks(monthly_prices, config),
    ),
    "Breakout": (
        select_breakout_stocks(monthly_prices, config),
        select_breakdown_stocks(monthly_prices, config),
    ),
}

In [ ]:
# =====================================================
# RUN: LONG-ONLY vs REGIME-GATED LONG/SHORT
# =====================================================
# Positions shifted by one month so we never trade on same-day info.

results = {}

for name, (long_signal, short_signal) in signal_pairs.items():
    long_positions = long_signal.shift(1)
    short_positions = short_signal.shift(1)

    # Baseline: long-only, no regime filter (same as Sprint 4)
    _, long_only_growth = build_portfolio(monthly_returns, long_positions)

    # New: regime-gated long/short
    gated_longs = apply_regime_filter(long_positions, regime, direction="long")
    gated_shorts = apply_regime_filter(short_positions, regime, direction="short")
    _, long_short_growth = build_long_short_portfolio(monthly_returns, gated_longs, gated_shorts)

    results[name] = {
        "long_only": long_only_growth,
        "regime_long_short": long_short_growth,
    }

In [ ]:
# =====================================================
# PLOT: LONG-ONLY vs REGIME-GATED LONG/SHORT, PER STRATEGY
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (name, r) in zip(axes.flat, results.items()):
    r["long_only"].plot(ax=ax, label="Long only")
    r["regime_long_short"].plot(ax=ax, label="Regime-gated long/short")
    ax.set_title(name)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# NOTES
# =====================================================
# build_long_short_portfolio assumes an equal split of capital between
# the long leg and the short leg - a simplification. Real position
# sizing against your 1%-per-trade rule is Sprint 10's job.
#
# The regime filter uses a single benchmark (config.regime_benchmark,
# default SPY) and a single moving-average window. It's a blunt
# instrument on purpose - the point is to stop trading against a
# strongly trending market, not to time every wiggle.